In [1]:
import pandas as pd
df = pd.read_csv('Covid19_bal.csv')
from sklearn.model_selection import train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['title'], 
    df['label'],            
    test_size=0.2,               
    random_state=42,               
)

In [2]:
print("Train Samples:", len(X_train_text))
print("Test Samples :", len(X_test_text))

Train Samples: 7846
Test Samples : 1962


In [3]:
# =========================
# BERT-Base + Hyperparameter Tuning Pipeline
# =========================

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

# ----------------------------------------------------------
# LOAD BERT
# ----------------------------------------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert_model.to(device)
bert_model.eval()

print("Device:", device)

# ----------------------------------------------------------
# DATASET CLASS
# ----------------------------------------------------------
class TextDataset(Dataset):

    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

# ----------------------------------------------------------
# FUNCTION TO EXTRACT BERT EMBEDDINGS
# ----------------------------------------------------------
def get_bert_embeddings(texts, batch_size=16):

    dataset = TextDataset(texts, tokenizer)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    embeddings = []

    with torch.no_grad():

        for batch in tqdm(loader, desc="Generating BERT Embeddings"):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = bert_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            # FIXED: Use pooler_output instead of raw CLS token
            cls_embedding = outputs.pooler_output

            embeddings.append(cls_embedding.cpu().numpy())

    return np.vstack(embeddings)

# ----------------------------------------------------------
# EXTRACT TRAIN / TEST EMBEDDINGS
# ----------------------------------------------------------
print("\nExtracting Train Embeddings...")
X_train = get_bert_embeddings(X_train_text)

print("\nExtracting Test Embeddings...")
X_test = get_bert_embeddings(X_test_text)

print("\nTrain Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

# ----------------------------------------------------------
# MODELS + PARAM GRID
# ----------------------------------------------------------
param_grid = {
    "LR": {
        "model": LogisticRegression(max_iter=1000, random_state=42),
        "params": {"C": [0.1, 0.5, 1, 2, 5]}
    },

    "SVM Linear": {
        "model": SVC(kernel="linear", random_state=42),
        "params": {"C": [0.1, 0.5, 1, 2, 5]}
    },

    "SVM RBF": {
        "model": SVC(kernel="rbf", random_state=42),
        "params": {
            "C": [0.5, 1, 2, 5],
            "gamma": ["scale", "auto", 0.1]
        }
    },

    "RF": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200, 300],
            "max_depth": [10, 20, None],
            "min_samples_split": [2, 5, 10]
        }
    }
}

print("\nMODEL SELECTION WITH PROPER 5-FOLD CROSS VALIDATION")

# ----------------------------------------------------------
# STEP 1: HYPERPARAMETER TUNING
# ----------------------------------------------------------
results = []
best_models = {}

for model_name, model_params in param_grid.items():

    print("\nTuning:", model_name)

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    grid = GridSearchCV(
        estimator=model_params["model"],
        param_grid=model_params["params"],
        cv=skf,
        scoring="accuracy",
        n_jobs=-1,
        verbose=0
    )

    grid.fit(X_train, y_train)

    best_models[model_name] = grid.best_estimator_

    # Cross-validated predictions
    y_cv_pred = cross_val_predict(
        grid.best_estimator_,
        X_train,
        y_train,
        cv=skf,
        n_jobs=-1
    )

    cv_acc = accuracy_score(y_train, y_cv_pred)
    cv_prec = precision_score(y_train, y_cv_pred, average="binary")
    cv_rec = recall_score(y_train, y_cv_pred, average="binary")
    cv_f1 = f1_score(y_train, y_cv_pred, average="binary")

    results.append({
        "Model": model_name,
        "Best Params": grid.best_params_,
        "CV Accuracy": round(cv_acc, 4),
        "CV Precision": round(cv_prec, 4),
        "CV Recall": round(cv_rec, 4),
        "CV F1": round(cv_f1, 4)
    })

    print("Best Params:", grid.best_params_)
    print("CV Accuracy :", round(cv_acc, 4))
    print("CV Precision:", round(cv_prec, 4))
    print("CV Recall   :", round(cv_rec, 4))
    print("CV F1 Score :", round(cv_f1, 4))

# ----------------------------------------------------------
# STEP 2: MODEL RANKING
# ----------------------------------------------------------
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(
    by="CV Accuracy",
    ascending=False
)

print("\nMODEL RANKING")
print(df_results)

best_model_name = df_results.iloc[0]["Model"]
best_model = best_models[best_model_name]

print("\nSELECTED BEST MODEL:", best_model_name)

# ----------------------------------------------------------
# STEP 3: RETRAIN BEST MODEL
# ----------------------------------------------------------
best_model.fit(X_train, y_train)

# ----------------------------------------------------------
# STEP 4: TEST EVALUATION
# ----------------------------------------------------------
print("\nFINAL TEST EVALUATION")

y_pred = best_model.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
test_prec = precision_score(y_test, y_pred, average="binary")
test_rec = recall_score(y_test, y_pred, average="binary")
test_f1 = f1_score(y_test, y_pred, average="binary")

print("Accuracy :", test_acc)
print("Precision:", test_prec)
print("Recall   :", test_rec)
print("F1 Score :", test_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ----------------------------------------------------------
# STEP 5: ALL MODELS TEST COMPARISON
# ----------------------------------------------------------
comparison_results = []

for model_name, estimator in best_models.items():

    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_test)

    comparison_results.append({
        "Model": model_name,
        "Test Accuracy": round(accuracy_score(y_test, pred), 4),
        "Test Precision": round(precision_score(y_test, pred), 4),
        "Test Recall": round(recall_score(y_test, pred), 4),
        "Test F1": round(f1_score(y_test, pred), 4)
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values(
    by=["Test F1", "Test Accuracy"],
    ascending=False
)

print("\nALL MODELS TEST PERFORMANCE")
print(comparison_df)

best_test = comparison_df.iloc[0]

print("\nBEST TEST MODEL:", best_test["Model"])
print("Accuracy :", best_test["Test Accuracy"])
print("Precision:", best_test["Test Precision"])
print("Recall   :", best_test["Test Recall"])
print("F1 Score :", best_test["Test F1"])

Device: cuda

Extracting Train Embeddings...


Generating BERT Embeddings:   0%|          | 0/491 [00:00<?, ?it/s]


Extracting Test Embeddings...


Generating BERT Embeddings:   0%|          | 0/123 [00:00<?, ?it/s]


Train Shape: (7846, 768)
Test Shape : (1962, 768)

MODEL SELECTION WITH PROPER 5-FOLD CROSS VALIDATION

Tuning: LR
Best Params: {'C': 5}
CV Accuracy : 0.9344
CV Precision: 0.944
CV Recall   : 0.9236
CV F1 Score : 0.9337

Tuning: SVM Linear
Best Params: {'C': 5}
CV Accuracy : 0.9369
CV Precision: 0.9438
CV Recall   : 0.9292
CV F1 Score : 0.9364

Tuning: SVM RBF
Best Params: {'C': 5, 'gamma': 0.1}
CV Accuracy : 0.9337
CV Precision: 0.9416
CV Recall   : 0.9248
CV F1 Score : 0.9332

Tuning: RF
Best Params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 300}
CV Accuracy : 0.8898
CV Precision: 0.9192
CV Recall   : 0.8548
CV F1 Score : 0.8858

MODEL RANKING
        Model                                        Best Params  CV Accuracy  \
1  SVM Linear                                           {'C': 5}       0.9369   
0          LR                                           {'C': 5}       0.9344   
2     SVM RBF                             {'C': 5, 'gamma': 0.1}       0.9337   
3  

In [4]:
import pickle

with open("best_bert_embed_covid.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Best model saved successfully")

Best model saved successfully
